# The two axes of agent modes

When designing a mode for an AI agent, there are two independent questions to answer:
1. **How autonomous should the agent be?** - Should it always ask before acting, act freely within a scope, or operate completely independently?
2. **What kind of work should the agent do?** - Should it execute tasks, research information, produce plans, critique outputs, or monitor a system?

These two questions define two orthogonal axes that together characterize any agent mode. They are independent: a highly autonomous agent can do research, task execution, or planning; a minimally autonomous agent can similarly adopt any behavioral type.

**Axis 1 — Autonomy level (0–4)**:
- Level 0: Chat - agent informs, human decides and acts on everything.
- Level 1: Copilot - agent suggests, human approves and executes.
- Level 2: Supervised - agent executes with human approval at defined checkpoints.
- Level 3: Semi-autonomous - agent executes low-risk actions, escalates high-risk ones.
- Level 4: Fully autonomous - agent executes, monitors, and self-corrects without interruption.

**Axis 2 — Behavioral type**:
- Type A: Task Execution - goal-directed work, producing deliverables.
- Type B: Research - information gathering, synthesis, citation.
- Type C: Planning - decomposing objectives into structured plans.
- Type D: Critic/Reflection - evaluating, finding flaws, improving.
- Type E: Conversational - dialogue-focused, responsive, explanatory.
- Type F: Monitoring - passive observation, alerting on conditions.

Any combination of these axes is valid. A `Research + Supervised` mode gives us an agent that gathers information autonomously but requires human sign-off before producing its final report. A `Task Execution + Fully Autonomous` mode gives us an agent that completes deliverables end-to-end.

In this notebook, we implement both axes with typed data structures, demonstrate the behavioral difference along each axis, and show how to compose any autonomy-behavioral combination into a runtime mode.

In [1]:
import os
from dataclasses import dataclass
from enum import Enum
from typing import List, Optional
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

### Initialize the language model

In [2]:
llm = ChatOpenAI(model='gpt-4o-mini', api_key=os.getenv("OPENAI_API_KEY", "").strip(), temperature=0)

## Axis 1: Autonomy level
The autonomy axis answers: *how independently does the agent act?* Let's implement it as a typed enum with associated behavioral instructions:

In [3]:
@dataclass(frozen=True)
class AutonomyLevel:
    """Defines how independently an agent acts within a session.

    Args:
        level: Integer 0-4 indicating autonomy (0=passive, 4=fully autonomous).
        name: Human-readable label.
        decision_rule: Who makes final decisions.
        execution_rule: Who executes state-changing actions.
        escalation_trigger: Conditions that trigger a pause.
        behavioral_instruction: System-prompt text encoding this autonomy level.
    """
    level: int
    name: str
    decision_rule: str
    execution_rule: str
    escalation_trigger: str
    behavioral_instruction: str


# The five autonomy levels
AUTONOMY_CHAT = AutonomyLevel(
    level=0,
    name="Chat",
    decision_rule="Human decides everything",
    execution_rule="Human executes all actions",
    escalation_trigger="Always — agent only provides information",
    behavioral_instruction=(
        "You are in Chat mode (autonomy level 0). "
        "Provide information and explanations only. "
        "Never suggest that you will take any action. Use 'you should' not 'I will'."
    ),
)

AUTONOMY_COPILOT = AutonomyLevel(
    level=1,
    name="Copilot",
    decision_rule="Agent recommends, human decides",
    execution_rule="Human executes all actions",
    escalation_trigger="Any state-changing action",
    behavioral_instruction=(
        "You are in Copilot mode (autonomy level 1). "
        "Suggest actions with clear rationale and expected outcomes. "
        "Present the recommendation, then wait. Do not execute; the human will."
    ),
)

AUTONOMY_SUPERVISED = AutonomyLevel(
    level=2,
    name="Supervised",
    decision_rule="Agent decides within scope; checkpoints for high-impact actions",
    execution_rule="Agent executes after checkpoint approval",
    escalation_trigger="Predefined high-impact checkpoints",
    behavioral_instruction=(
        "You are in Supervised mode (autonomy level 2). "
        "Execute low-impact steps autonomously. "
        "Before any high-impact action, present a summary and explicitly request approval: "
        "'Ready to proceed — please confirm.'"
    ),
)

AUTONOMY_SEMI_AUTO = AutonomyLevel(
    level=3,
    name="Semi-Autonomous",
    decision_rule="Agent decides; escalates only on uncertainty or high risk",
    execution_rule="Agent executes all low/medium risk actions",
    escalation_trigger="High-risk or irreversible actions only",
    behavioral_instruction=(
        "You are in Semi-Autonomous mode (autonomy level 3). "
        "Execute tasks without asking for confirmation unless an action is irreversible or high-risk. "
        "Report what you did after each significant step."
    ),
)

AUTONOMY_FULL = AutonomyLevel(
    level=4,
    name="Fully Autonomous",
    decision_rule="Agent decides and self-corrects",
    execution_rule="Agent executes all actions",
    escalation_trigger="Never — agent handles all cases",
    behavioral_instruction=(
        "You are in Fully Autonomous mode (autonomy level 4). "
        "Complete the objective end-to-end without asking for input. "
        "If you encounter issues, adapt and self-correct. Report outcomes when done."
    ),
)

ALL_AUTONOMY_LEVELS = [AUTONOMY_CHAT, AUTONOMY_COPILOT, AUTONOMY_SUPERVISED, AUTONOMY_SEMI_AUTO, AUTONOMY_FULL]

print(f"{'Level':<6} {'Name':<18} {'Decision Rule':<42} {'Escalates On'}")
print("-" * 100)
for a in ALL_AUTONOMY_LEVELS:
    print(f"{a.level:<6} {a.name:<18} {a.decision_rule:<42} {a.escalation_trigger}")

Level  Name               Decision Rule                              Escalates On
----------------------------------------------------------------------------------------------------
0      Chat               Human decides everything                   Always — agent only provides information
1      Copilot            Agent recommends, human decides            Any state-changing action
2      Supervised         Agent decides within scope; checkpoints for high-impact actions Predefined high-impact checkpoints
3      Semi-Autonomous    Agent decides; escalates only on uncertainty or high risk High-risk or irreversible actions only
4      Fully Autonomous   Agent decides and self-corrects            Never — agent handles all cases


## Demonstrating the autonomy axis
Let's run the same task - "there's a memory leak in our production service" - across four autonomy levels. Watch how the interaction style fundamentally changes, even though the underlying task is identical:

In [4]:
def run_with_autonomy(task: str, autonomy: AutonomyLevel) -> None:
    """Run a task under a given autonomy level and print the result.

    Args:
        task: The operational task to perform.
        autonomy: The autonomy level to apply.
    """
    print(f"=== Autonomy Level {autonomy.level}: {autonomy.name} ===")
    response = llm.invoke([
        SystemMessage(content=autonomy.behavioral_instruction),
        HumanMessage(content=task),
    ])
    print(response.content)
    print()


incident_task = (
    "Our production API service has a memory leak — heap usage grows by 50MB/hour "
    "and the service crashes every 6 hours. We need this addressed."
)

# Show autonomy levels 0, 1, 2, and 4 for contrast
run_with_autonomy(incident_task, AUTONOMY_CHAT)
run_with_autonomy(incident_task, AUTONOMY_COPILOT)
run_with_autonomy(incident_task, AUTONOMY_SUPERVISED)
run_with_autonomy(incident_task, AUTONOMY_FULL)

=== Autonomy Level 0: Chat ===
Addressing a memory leak in your production API service is crucial to maintaining stability and performance. Here are steps you should consider to identify and resolve the issue:

1. **Monitor Memory Usage**: Use monitoring tools to track memory usage over time. Tools like Prometheus, Grafana, or APM solutions (e.g., New Relic, Datadog) can help visualize memory trends and identify when the leak occurs.

2. **Profile the Application**: Use a memory profiler to analyze heap dumps. Tools like VisualVM, YourKit, or Eclipse Memory Analyzer (MAT) can help you identify objects that are not being released and are contributing to the memory leak.

3. **Analyze Heap Dumps**: Capture heap dumps at regular intervals (e.g., every hour) and analyze them to find out which objects are consuming the most memory. Look for objects that should have been garbage collected but are still in memory.

4. **Review Code for Common Issues**: Check for common causes of memory leaks,

Each autonomy level produced a distinctly different response:
- **Chat (0)**: Purely informational — "you should check", "you should monitor".
- **Copilot (1)**: Concrete recommendation with rationale, but leaves execution to the human.
- **Supervised (2)**: Takes ownership of steps, identifies the checkpoint before restart.
- **Fully Autonomous (4)**: Plans and commits to full execution — "I will diagnose, patch, and verify".

Same task. Same model. Same information. Completely different interaction pattern.

## Axis 2: Behavioral type
The behavioral type axis answers: *what kind of work is the agent optimized for?* This is independent of autonomy - a research agent can be fully autonomous or completely supervised.

Let's implement behavioral types as typed structures:

In [5]:
@dataclass(frozen=True)
class BehavioralType:
    """Defines the primary work pattern an agent is optimized for.

    Args:
        type_id: Short identifier (A-F).
        name: Human-readable label.
        primary_goal: What the agent optimizes for.
        output_shape: What the agent's responses typically look like.
        behavioral_instruction: System-prompt text encoding this behavioral type.
    """
    type_id: str
    name: str
    primary_goal: str
    output_shape: str
    behavioral_instruction: str


BEHAVIORAL_TASK_EXECUTION = BehavioralType(
    type_id="A",
    name="Task Execution",
    primary_goal="Produce a concrete deliverable",
    output_shape="Code, document, structured artifact",
    behavioral_instruction=(
        "You are a Task Execution agent. Your goal is to produce a concrete deliverable. "
        "Be direct and output-oriented. Produce working artifacts, not discussions. "
        "Minimize preamble; maximize output quality."
    ),
)

BEHAVIORAL_RESEARCH = BehavioralType(
    type_id="B",
    name="Research",
    primary_goal="Gather, synthesize, and cite information",
    output_shape="Synthesis with sources, uncertainty flagged",
    behavioral_instruction=(
        "You are a Research agent. Your goal is to gather and synthesize information thoroughly. "
        "Cite evidence for every claim. Flag what is uncertain or requires verification. "
        "Do not recommend actions — present findings and let the human decide."
    ),
)

BEHAVIORAL_PLANNING = BehavioralType(
    type_id="C",
    name="Planning",
    primary_goal="Decompose an objective into an actionable plan",
    output_shape="Structured plan with steps, owners, dependencies",
    behavioral_instruction=(
        "You are a Planning agent. Your goal is to decompose an objective into a structured plan. "
        "Output phases, steps, dependencies, and success criteria. "
        "Do not execute steps — produce the plan for human review."
    ),
)

BEHAVIORAL_CRITIC = BehavioralType(
    type_id="D",
    name="Critic/Reflection",
    primary_goal="Identify flaws, risks, and improvements",
    output_shape="Structured critique with severity ratings",
    behavioral_instruction=(
        "You are a Critic agent. Your goal is to find problems, not validate choices. "
        "For every artifact or plan presented, identify weaknesses, risks, and gaps. "
        "Rate each finding by severity (Critical/High/Medium/Low). Be direct and concise."
    ),
)

BEHAVIORAL_CONVERSATIONAL = BehavioralType(
    type_id="E",
    name="Conversational",
    primary_goal="Guide the human through understanding via dialogue",
    output_shape="Responsive, explanatory, asks clarifying questions",
    behavioral_instruction=(
        "You are a Conversational agent. Your goal is to help the human understand through dialogue. "
        "Ask clarifying questions when needed. Build understanding incrementally. "
        "Adapt your explanations to the human's apparent level of familiarity."
    ),
)

BEHAVIORAL_MONITORING = BehavioralType(
    type_id="F",
    name="Monitoring",
    primary_goal="Observe conditions and alert on anomalies",
    output_shape="Status reports and anomaly alerts",
    behavioral_instruction=(
        "You are a Monitoring agent. Your goal is passive observation and alerting. "
        "Assess the described system state and report: Normal / Warning / Critical. "
        "Flag anomalies with their threshold violation and recommended escalation path."
    ),
)

ALL_BEHAVIORAL_TYPES = [
    BEHAVIORAL_TASK_EXECUTION, BEHAVIORAL_RESEARCH, BEHAVIORAL_PLANNING,
    BEHAVIORAL_CRITIC, BEHAVIORAL_CONVERSATIONAL, BEHAVIORAL_MONITORING
]

print(f"{'Type':<6} {'Name':<20} {'Primary Goal':<40} {'Output Shape'}")
print("-" * 105)
for b in ALL_BEHAVIORAL_TYPES:
    print(f"{b.type_id:<6} {b.name:<20} {b.primary_goal:<40} {b.output_shape}")

Type   Name                 Primary Goal                             Output Shape
---------------------------------------------------------------------------------------------------------
A      Task Execution       Produce a concrete deliverable           Code, document, structured artifact
B      Research             Gather, synthesize, and cite information Synthesis with sources, uncertainty flagged
C      Planning             Decompose an objective into an actionable plan Structured plan with steps, owners, dependencies
D      Critic/Reflection    Identify flaws, risks, and improvements  Structured critique with severity ratings
E      Conversational       Guide the human through understanding via dialogue Responsive, explanatory, asks clarifying questions
F      Monitoring           Observe conditions and alert on anomalies Status reports and anomaly alerts


## Demonstrating the behavioral axis
Same question - "how should we handle the API memory leak?" - under four different behavioral types. Notice that the autonomy level stays constant (Copilot, level 1), but the output structure and intent change entirely:

In [6]:
def run_with_behavioral_type(task: str, behavioral: BehavioralType, autonomy: AutonomyLevel) -> None:
    """Run a task under a specific behavioral type with a fixed autonomy level.

    Args:
        task: The problem or request to address.
        behavioral: The behavioral type defining the agent's work pattern.
        autonomy: The autonomy level (held constant in this demo).
    """
    print(f"=== Type {behavioral.type_id}: {behavioral.name} (autonomy: {autonomy.name}) ===")
    # Compose both axes into the system prompt
    system_prompt = f"{behavioral.behavioral_instruction}\n\n{autonomy.behavioral_instruction}"
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=task),
    ])
    print(response.content)
    print()


# Hold autonomy at Copilot (level 1); vary behavioral type
run_with_behavioral_type(incident_task, BEHAVIORAL_RESEARCH, AUTONOMY_COPILOT)
run_with_behavioral_type(incident_task, BEHAVIORAL_PLANNING, AUTONOMY_COPILOT)
run_with_behavioral_type(incident_task, BEHAVIORAL_CRITIC, AUTONOMY_COPILOT)
run_with_behavioral_type(incident_task, BEHAVIORAL_MONITORING, AUTONOMY_COPILOT)

=== Type B: Research (autonomy: Copilot) ===
To address the memory leak in your production API service, it is essential to gather information on potential causes, identify tools for diagnosis, and outline steps for resolution. Here are the findings:

### Potential Causes of Memory Leaks
1. **Unreleased Resources**: Objects that are no longer needed but are still referenced can lead to memory leaks. This often occurs with event listeners, callbacks, or global variables that are not properly cleaned up (Morrison, 2020).
2. **Circular References**: In languages with garbage collection, circular references can prevent memory from being freed. This is particularly common in JavaScript and Java (Harris, 2021).
3. **Large Data Structures**: Accumulating large data structures without proper management can lead to increased memory usage (Smith, 2022).
4. **Third-party Libraries**: Sometimes, memory leaks can originate from external libraries that are not well-maintained or have known issues (Jo

Each behavioral type produces a fundamentally different kind of output:
- **Research**: Gathers information, cites evidence, surfaces uncertainty.
- **Planning**: Decomposes into phases, steps, dependencies.
- **Critic**: Identifies risks and gaps, rates severity.
- **Monitoring**: Assesses current state, issues status, flags thresholds.

Same autonomy level, completely different work output.

## Composing the axes: Building any mode
Now the key insight: we can combine any autonomy level with any behavioral type to create a specific mode. The function below takes one value from each axis and composes them into a complete mode system prompt:

In [7]:
def compose_mode(autonomy: AutonomyLevel, behavioral: BehavioralType) -> str:
    """Compose a complete mode system prompt from autonomy and behavioral axes.

    Args:
        autonomy: The autonomy level for this mode.
        behavioral: The behavioral type for this mode.

    Returns:
        A complete system prompt encoding the composed mode.
    """
    return (
        f"MODE: {behavioral.name} + {autonomy.name} "
        f"(Behavioral Type {behavioral.type_id}, Autonomy Level {autonomy.level})\n\n"
        f"Behavioral contract:\n"
        f"{behavioral.behavioral_instruction}\n\n"
        f"Autonomy contract:\n"
        f"{autonomy.behavioral_instruction}"
    )


def run_composed_mode(task: str, autonomy: AutonomyLevel, behavioral: BehavioralType) -> None:
    """Run a task using a mode composed from both axes.

    Args:
        task: The task to perform.
        autonomy: Autonomy level.
        behavioral: Behavioral type.
    """
    mode_name = f"{behavioral.name} + {autonomy.name}"
    print(f"{'=' * 60}")
    print(f"MODE: {mode_name}")
    print(f"{'=' * 60}")

    system_prompt = compose_mode(autonomy, behavioral)
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=task),
    ])
    print(response.content)
    print()


# Demo 1: Research + Supervised
# Good for: information gathering tasks where you want human sign-off before reporting conclusions
run_composed_mode(
    "Investigate whether our current Redis configuration can handle 10x the current load.",
    autonomy=AUTONOMY_SUPERVISED,
    behavioral=BEHAVIORAL_RESEARCH,
)

# Demo 2: Task Execution + Semi-Autonomous
# Good for: delivery-focused tasks where the agent handles low-risk steps but escalates before major deployments
run_composed_mode(
    "Create a Python script that monitors Redis memory usage and alerts when it exceeds 80%.",
    autonomy=AUTONOMY_SEMI_AUTO,
    behavioral=BEHAVIORAL_TASK_EXECUTION,
)

MODE: Research + Supervised
To determine if your current Redis configuration can handle 10 times the current load, several factors need to be considered, including hardware specifications, Redis configuration settings, and the nature of the workload. Here’s a structured approach to gather relevant information:

### 1. **Current Redis Configuration**
   - **Memory Usage**: Check the current memory usage and limits. Redis is an in-memory data store, and its performance is heavily dependent on available memory.
   - **Persistence Settings**: Review the persistence settings (RDB, AOF) as they can impact performance under heavy load.
   - **Data Structure Usage**: Analyze the types of data structures being used (strings, lists, sets, hashes) and their efficiency under load.
   - **Replication and Sharding**: If using replication or sharding, assess how these configurations are set up and their impact on performance.

### 2. **Hardware Specifications**
   - **CPU and RAM**: Evaluate the curr

## Mode Selection: A decision framework
Given that there are 5 autonomy levels × 6 behavioral types = 30 possible mode combinations, how do we pick the right one? Here is a decision framework:

In [8]:
def select_mode(
    user_retains_execution: bool,
    task_is_exploratory: bool,
    needs_structured_output: bool,
    has_destructive_actions: bool,
    is_established_workflow: bool,
) -> tuple[AutonomyLevel, BehavioralType]:
    """Select the appropriate mode based on task characteristics.

    Args:
        user_retains_execution: Human will execute all state-changing actions.
        task_is_exploratory: Task is open-ended research/discovery.
        needs_structured_output: Task requires a specific deliverable format.
        has_destructive_actions: Task involves deletes, drops, or irreversible changes.
        is_established_workflow: The workflow has been validated and is well-understood.

    Returns:
        Tuple of (AutonomyLevel, BehavioralType) for the recommended mode.
    """
    # Select autonomy level
    if user_retains_execution:
        autonomy = AUTONOMY_COPILOT
    elif has_destructive_actions and not is_established_workflow:
        autonomy = AUTONOMY_SUPERVISED
    elif has_destructive_actions and is_established_workflow:
        autonomy = AUTONOMY_SEMI_AUTO
    elif is_established_workflow:
        autonomy = AUTONOMY_FULL
    else:
        autonomy = AUTONOMY_SUPERVISED

    # Select behavioral type
    if task_is_exploratory:
        behavioral = BEHAVIORAL_RESEARCH
    elif needs_structured_output:
        behavioral = BEHAVIORAL_TASK_EXECUTION
    else:
        behavioral = BEHAVIORAL_PLANNING

    return autonomy, behavioral


# Test the decision framework on three scenarios
SCENARIOS = [
    {
        "name": "Exploring an unfamiliar codebase",
        "user_retains_execution": True,
        "task_is_exploratory": True,
        "needs_structured_output": False,
        "has_destructive_actions": False,
        "is_established_workflow": False,
    },
    {
        "name": "Generating a weekly performance report",
        "user_retains_execution": False,
        "task_is_exploratory": False,
        "needs_structured_output": True,
        "has_destructive_actions": False,
        "is_established_workflow": True,
    },
    {
        "name": "Executing a database schema migration",
        "user_retains_execution": False,
        "task_is_exploratory": False,
        "needs_structured_output": True,
        "has_destructive_actions": True,
        "is_established_workflow": False,
    },
]

print("Mode Selection Results")
print("=" * 60)
for scenario in SCENARIOS:
    name = scenario.pop("name")
    autonomy, behavioral = select_mode(**scenario)
    print(f"Scenario: {name}")
    print(f"  Recommended: {behavioral.name} + {autonomy.name} (Type {behavioral.type_id}, Level {autonomy.level})")
    print(f"  Rationale: Behavioral={behavioral.primary_goal}; Autonomy={autonomy.decision_rule}")
    print()

Mode Selection Results
Scenario: Exploring an unfamiliar codebase
  Recommended: Research + Copilot (Type B, Level 1)
  Rationale: Behavioral=Gather, synthesize, and cite information; Autonomy=Agent recommends, human decides

Scenario: Generating a weekly performance report
  Recommended: Task Execution + Fully Autonomous (Type A, Level 4)
  Rationale: Behavioral=Produce a concrete deliverable; Autonomy=Agent decides and self-corrects

Scenario: Executing a database schema migration
  Recommended: Task Execution + Supervised (Type A, Level 2)
  Rationale: Behavioral=Produce a concrete deliverable; Autonomy=Agent decides within scope; checkpoints for high-impact actions



The decision framework surfaces sensible combinations:
- Exploratory + user-controlled → **Research + copilot** (agent gathers info, human makes all calls).
- Recurring report → **Task execution + fully autonomous** (established workflow, structured output, no destruction).
- Schema migration → **Task execution + supervised** (structured deliverable, but checkpoint before destructive step).

1. **Autonomy level is about trust, not capability**: A research agent at level 4 (fully autonomous) is not more capable than one at level 1 (copilot) - it just operates without human checkpoints. The trust required to deploy at level 4 must be earned.
2. **Behavioral type shapes the output structure**: The same information gathered under research mode and task execution mode produces completely different outputs - one is a synthesis, the other is a deliverable.
3. **Start conservative on autonomy**: When in doubt, choose one level below where we think we need. A Supervised agent that occasionally needs an extra click to approve is far safer than a semi-autonomous agent that makes the wrong call unsupervised.